# Customer Churn — Data Cleaning and Preprocessing

**Phase 2 scope:** apply conservative deterministic cleaning, preserve an auditable cleaned CSV, create the fixed stratified split, and define leakage-safe preprocessing. This notebook does not perform advanced EDA, feature engineering, model training, tuning, or final-test evaluation.

## Data-protection rules

The raw CSV is read only. No observation is silently removed, missing values remain missing in the cleaned CSV, and statistical outliers remain unchanged. Learned imputation, scaling, and encoding are fitted only on training rows. `customer_id` is retained separately for traceability and never enters the feature matrix; `churn` is the target and also never enters the transformer.

## 1. Setup and project paths

The path search supports running from either the repository root or the `notebooks` directory. Reusable logic is imported from `src/preprocessing.py` rather than duplicated here.

In [1]:
from pathlib import Path
import hashlib
import sys

import numpy as np
import pandas as pd
from IPython.display import display

candidate_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path for path in candidate_roots if (path / "src" / "preprocessing.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root or src/preprocessing.py.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import (
    IDENTIFIER_COLUMN,
    TARGET_COLUMN,
    apply_deterministic_cleaning,
    build_preprocessing_transformer,
    create_or_load_fixed_split,
    get_feature_groups,
    load_and_validate_data,
    separate_features_target_ids,
)

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "customer_churn_advanced_dataset.csv"
CLEANED_PATH = PROJECT_ROOT / "data" / "processed" / "02_cleaned_dataset.csv"
SPLIT_PATH = PROJECT_ROOT / "data" / "processed" / "02_split_indices.json"
AUDIT_PATH = PROJECT_ROOT / "reports" / "data_audit.md"
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## 2. Review the Phase 1 audit and load the raw data

Phase 1 found 1,200 rows and 20 columns; 138 missing cells across 132 rows; no complete duplicates or repeated customer IDs; and a target distribution of 164 class-0 and 1,036 class-1 records. It also flagged unusual values and cross-field questions but found no definite error that justified changing a customer record. The code confirms that the report exists and validates the source against the audited schema.

In [ ]:
if not AUDIT_PATH.is_file():
    raise FileNotFoundError("Phase 1 report is missing; stop before Phase 2.")

raw_sha256_before = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()
raw_df = load_and_validate_data(RAW_PATH)

print(f"Loaded: {RAW_PATH}")
print(f"Shape: {raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns")
print(f"Raw SHA-256: {raw_sha256_before}")
display(raw_df.head())

## 3. Missing values

Missing measurements are genuine unknowns in the current file. Replacing them before splitting would let the final test set influence learned medians. They are therefore measured here, preserved in the cleaned CSV, and handled later inside the preprocessing pipeline.

In [3]:
missing_summary = pd.DataFrame({
    "missing_count": raw_df.isna().sum(),
    "missing_pct": raw_df.isna().mean().mul(100),
}).query("missing_count > 0").sort_values("missing_count", ascending=False)
display(missing_summary)
print(f"Rows with any missing value: {raw_df.isna().any(axis=1).sum():,}")
print(f"Total missing cells: {raw_df.isna().sum().sum():,}")

,missing_count,missing_pct
satisfaction_score,48,4.00
monthly_usage_gb,36,3.00
monthly_charges,30,2.50
total_charges,24,2.00


Rows with any missing value: 132
Total missing cells: 138


## 4. Duplicate and split-integrity checks

Exact duplicate rows can overweight observations, while repeated customer IDs could place the same customer in both partitions. Neither type is dropped silently. The reusable validator stops with an informative error if normalized customer IDs repeat.

In [4]:
duplicate_checks = pd.Series({
    "complete_duplicate_rows": int(raw_df.duplicated().sum()),
    "missing_customer_ids": int(raw_df[IDENTIFIER_COLUMN].isna().sum()),
    "duplicate_customer_ids_beyond_first": int(raw_df[IDENTIFIER_COLUMN].duplicated().sum()),
    "distinct_customer_ids": int(raw_df[IDENTIFIER_COLUMN].nunique()),
}, name="count")
display(duplicate_checks.to_frame())

,count
complete_duplicate_rows,0
missing_customer_ids,0
duplicate_customer_ids_beyond_first,0
distinct_customer_ids,1200


## 5. Numerical and categorical validation

The loader already validates numeric convertibility, binary target labels, non-negative values where negative values are inherently invalid, satisfaction on the observed 0–10 scale, and discount percentages from 0–100. These broad constraints do not invent a senior-age rule or narrow business limits. Text checks look only for accidental blanks or surrounding whitespace.

In [5]:
numerical_features, categorical_features = get_feature_groups(raw_df)
print(f"Numerical predictors ({len(numerical_features)}): {numerical_features}")
print(f"Categorical predictors ({len(categorical_features)}): {categorical_features}")
display(raw_df[numerical_features].describe().T[["count", "min", "50%", "max"]])

text_features = [column for column in categorical_features if column != "senior_citizen"]
text_quality = []
for column in text_features:
    values = raw_df[column].astype("string")
    text_quality.append({
        "column": column,
        "categories": values.nunique(dropna=True),
        "blank_values": values.str.strip().eq("").sum(),
        "surrounding_whitespace": values.ne(values.str.strip()).sum(),
    })
display(pd.DataFrame(text_quality).set_index("column"))
print("Target labels and counts:")
display(raw_df[TARGET_COLUMN].value_counts().sort_index().rename("count").to_frame())

Numerical predictors (9): ['age', 'tenure_months', 'monthly_charges', 'total_charges', 'complaints_last_12m', 'late_payments_last_12m', 'satisfaction_score', 'monthly_usage_gb', 'discount_pct']
Categorical predictors (9): ['contract_type', 'internet_service', 'region', 'payment_method', 'paperless_billing', 'dependents', 'support_plan', 'tech_support', 'senior_citizen']


,count,min,50%,max
age,"1,200.00",18.00,47.00,75.00
tenure_months,"1,200.00",1.00,35.00,72.00
monthly_charges,"1,170.00",20.00,71.82,235.01
total_charges,"1,176.00",22.73,"2,244.46","8,853.76"
complaints_last_12m,"1,200.00",0.00,1.00,5.00
late_payments_last_12m,"1,200.00",0.00,1.00,4.00
satisfaction_score,"1,152.00",1.00,6.75,10.00
monthly_usage_gb,"1,164.00",2.00,66.40,466.90
discount_pct,"1,200.00",0.00,8.71,32.89


,categories,blank_values,surrounding_whitespace
column,,,
contract_type,3,0,0
internet_service,3,0,0
region,9,0,0
payment_method,4,0,0
paperless_billing,2,0,0
dependents,2,0,0
support_plan,2,0,0
tech_support,2,0,0


Target labels and counts:


,count
churn,
0,164
1,1036


## 6. Unusual values and IQR outliers

The 1.5 × IQR rule identifies statistically unusual values, not confirmed errors. It can over-flag low-frequency count data such as complaints. All flagged values remain unchanged; source verification is required before any future capping or exclusion decision.

In [6]:
outlier_flags = pd.DataFrame(False, index=raw_df.index, columns=numerical_features)
outlier_rows = []
for column in numerical_features:
    values = raw_df[column].dropna()
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    flags = raw_df[column].lt(lower) | raw_df[column].gt(upper)
    outlier_flags[column] = flags
    outlier_rows.append({
        "column": column,
        "lower_bound": lower,
        "upper_bound": upper,
        "flagged_count": int(flags.sum()),
    })
display(pd.DataFrame(outlier_rows).set_index("column"))
print(f"Unique rows flagged in at least one field: {outlier_flags.any(axis=1).sum():,}")
print("No outlier is removed or modified.")

,lower_bound,upper_bound,flagged_count
column,,,
age,-11.50,104.50,0
tenure_months,-36.00,108.00,0
monthly_charges,5.62,135.82,12
total_charges,"-2,829.97","7,701.22",14
complaints_last_12m,-1.50,2.50,41
late_payments_last_12m,-1.50,2.50,57
satisfaction_score,2.00,11.60,4
monthly_usage_gb,-31.05,161.55,5
discount_pct,-11.60,28.87,2


Unique rows flagged in at least one field: 125
No outlier is removed or modified.


## 7. Deterministic cleaning and cleaned CSV

The selected rules are deliberately conservative:

- trim surrounding whitespace in text fields because it can create duplicate categories;
- represent truly blank text as missing rather than as a false category;
- parse audited numerical fields without filling them;
- map unambiguous target labels `0 → 0` and `1 → 1`;
- reject repeated IDs, invalid targets, non-numeric text in numeric fields, and values outside broad documented screens;
- preserve all rows, genuine missing values, unusual values, and unresolved cross-field combinations.

Phase 1 found no whitespace or label problem in the actual file, so these representation rules are expected to change zero values.

In [7]:
cleaned_df = apply_deterministic_cleaning(raw_df)

raw_comparable = raw_df.astype("string").fillna("<MISSING>")
cleaned_comparable = cleaned_df.astype("string").fillna("<MISSING>")
changed_cells = int(raw_comparable.ne(cleaned_comparable).sum().sum())
changed_rows = int(raw_comparable.ne(cleaned_comparable).any(axis=1).sum())

CLEANED_PATH.parent.mkdir(parents=True, exist_ok=True)
cleaned_df.to_csv(CLEANED_PATH, index=False, lineterminator="\n")
reloaded_cleaned_df = load_and_validate_data(CLEANED_PATH)
cleaned_sha256 = hashlib.sha256(CLEANED_PATH.read_bytes()).hexdigest()
raw_sha256_after = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()

print(f"Saved: {CLEANED_PATH}")
print(f"Rows excluded: {len(raw_df) - len(reloaded_cleaned_df):,}")
print(f"Changed cells: {changed_cells:,}; changed rows: {changed_rows:,}")
print(f"Missing cells preserved: {reloaded_cleaned_df.isna().sum().sum():,}")
print(f"Cleaned SHA-256: {cleaned_sha256}")
print(f"Raw source unchanged: {raw_sha256_before == raw_sha256_after}")

Saved: D:\Internship Tasks\Task - 2026.10.07\Customer-Churn-Prediction\data\processed\02_cleaned_dataset.csv
Rows excluded: 0
Changed cells: 0; changed rows: 0
Missing cells preserved: 138
Cleaned SHA-256: b9f3981f7d32ba3bfd2256f69b6e12f730be79139b22dba6ced563f53db9462c
Raw source unchanged: True


**Interpretation:** no actual customer value required deterministic correction, so the cleaned file remains a validated, row-preserving copy. Its missing values are intentional inputs to the training-only pipeline. The unresolved charge/service questions and IQR flags are documented rather than guessed away.

## 8. Separate identifiers, target, and predictors

Separating roles before splitting prevents `customer_id` and `churn` from entering preprocessing. The ID Series stays aligned with rows for auditability and later customer-level outputs.

In [8]:
X, y, customer_ids = separate_features_target_ids(reloaded_cleaned_df)
assert IDENTIFIER_COLUMN not in X.columns
assert TARGET_COLUMN not in X.columns
print(f"Predictor matrix before transformation: {X.shape}")
print(f"Target shape: {y.shape}; ID shape: {customer_ids.shape}")
print(f"Predictor columns: {X.columns.tolist()}")

Predictor matrix before transformation: (1200, 18)
Target shape: (1200,); ID shape: (1200,)
Predictor columns: ['age', 'tenure_months', 'monthly_charges', 'total_charges', 'complaints_last_12m', 'late_payments_last_12m', 'satisfaction_score', 'monthly_usage_gb', 'discount_pct', 'contract_type', 'internet_service', 'region', 'payment_method', 'paperless_billing', 'dependents', 'support_plan', 'tech_support', 'senior_citizen']


## 9. Fixed 80/20 stratified split

The split uses `random_state=42` and stratifies on the validated binary target. Row indices and an order-sensitive dataset fingerprint are saved locally. A later run loads the same indices only when the fingerprint and settings match; changed data cause an error rather than a silent re-split. The final test rows are not used for fitting.

In [9]:
split = create_or_load_fixed_split(
    reloaded_cleaned_df,
    SPLIT_PATH,
    test_size=0.20,
    random_state=42,
)
train_indices = split["train_indices"]
test_indices = split["test_indices"]
X_train, X_test = X.iloc[train_indices].copy(), X.iloc[test_indices].copy()
y_train, y_test = y.iloc[train_indices].copy(), y.iloc[test_indices].copy()
ids_train = customer_ids.iloc[train_indices]
ids_test = customer_ids.iloc[test_indices]

split_distribution = pd.DataFrame({
    "full_pct": y.value_counts(normalize=True).sort_index().mul(100),
    "train_count": y_train.value_counts().sort_index(),
    "train_pct": y_train.value_counts(normalize=True).sort_index().mul(100),
    "test_count": y_test.value_counts().sort_index(),
    "test_pct": y_test.value_counts(normalize=True).sort_index().mul(100),
})
display(split_distribution)
print(f"Train size: {len(X_train):,}; test size: {len(X_test):,}")
print(f"Row-index overlap: {len(set(train_indices) & set(test_indices))}")
print(f"Customer-ID overlap: {len(set(ids_train) & set(ids_test))}")
print(f"Dataset fingerprint: {split['dataset_fingerprint']}")
print(f"Reused existing local split: {split['loaded_existing']}")

,full_pct,train_count,train_pct,test_count,test_pct
churn,,,,,
0,13.67,131,13.65,33,13.75
1,86.33,829,86.35,207,86.25


Train size: 960; test size: 240
Row-index overlap: 0
Customer-ID overlap: 0
Dataset fingerprint: df788deba598b73abdc3a41b8a1947af63a90c5f369c2a498386a2c0eefeb769
Reused existing local split: True


## 10. Leakage-safe preprocessing architecture

`build_preprocessing_transformer()` returns an **unfitted** `ColumnTransformer` for placement inside a future model `Pipeline`. Numerical predictors use median imputation followed by `StandardScaler`. If a numerical feature is entirely missing in a training fold, scikit-learn's `keep_empty_features=True` fallback retains it (with the documented zero fallback because no median exists). Categorical predictors use an explicit `Missing` level and dense one-hot encoding with `handle_unknown='ignore'`.

An explicit missing category is preferable here because it remains well-defined even if an entire categorical feature is missing in a training fold. No categorical predictor is missing in the current dataset.

In [10]:
preprocessor = build_preprocessing_transformer(numerical_features, categorical_features)
display(preprocessor)
print("The newly constructed transformer is unfitted. It will now be fitted on X_train only for validation.")

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{transformer_name}""``. See :meth:`str.format` method from the standard library for more info... versionadded:: 1.0.. versionchanged:: 1.6 `verbose_feature_names_out` can be a callable or a string to be formatted.",False
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per tran

The newly constructed transformer is unfitted. It will now be fitted on X_train only for validation.


## 11. Validate the transformer without evaluating a model

The transformer is fitted only on `X_train`; `X_test` is transformed using those learned training statistics. We inspect shapes, feature names, missingness, and learned medians—not predictive performance. The transformed arrays are not saved because future cross-validation must refit preprocessing inside every training fold.

In [11]:
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

numerical_imputer = preprocessor.named_transformers_["numeric"].named_steps["imputer"]
training_medians = pd.Series(
    numerical_imputer.statistics_,
    index=numerical_features,
    name="training_imputation_value",
)
display(training_medians.to_frame())
print(f"Prepared training shape: {X_train_prepared.shape}")
print(f"Prepared test shape: {X_test_prepared.shape}")
print(f"Output feature count: {len(feature_names)}")
print(f"Missing values after training transform: {np.isnan(X_train_prepared).sum():,}")
print(f"Missing values after test transform: {np.isnan(X_test_prepared).sum():,}")
assert X_train_prepared.shape[1] == X_test_prepared.shape[1]
assert not np.isnan(X_train_prepared).any()
assert not np.isnan(X_test_prepared).any()

,training_imputation_value
age,47.00
tenure_months,35.00
monthly_charges,71.21
total_charges,"2,175.62"
complaints_last_12m,1.00
late_payments_last_12m,1.00
satisfaction_score,6.80
monthly_usage_gb,65.00
discount_pct,8.55


Prepared training shape: (960, 38)
Prepared test shape: (240, 38)
Output feature count: 38
Missing values after training transform: 0
Missing values after test transform: 0


## 12. Before-and-after quality summary

This comparison makes every Phase 2 effect explicit. Missing-value counts remain unchanged in the persisted customer-level data because their treatment belongs inside the training pipeline.

In [12]:
quality_summary = pd.DataFrame({
    "measure": [
        "rows", "columns", "missing cells", "complete duplicates",
        "duplicate customer IDs", "changed cells", "IQR-flagged rows"
    ],
    "before": [
        len(raw_df), raw_df.shape[1], raw_df.isna().sum().sum(), raw_df.duplicated().sum(),
        raw_df[IDENTIFIER_COLUMN].duplicated().sum(), 0, outlier_flags.any(axis=1).sum(),
    ],
    "after_cleaned_csv": [
        len(reloaded_cleaned_df), reloaded_cleaned_df.shape[1],
        reloaded_cleaned_df.isna().sum().sum(), reloaded_cleaned_df.duplicated().sum(),
        reloaded_cleaned_df[IDENTIFIER_COLUMN].duplicated().sum(), changed_cells,
        outlier_flags.any(axis=1).sum(),
    ],
    "decision": [
        "preserved", "preserved", "preserved for pipeline", "none found",
        "none found", "no correction justified", "retained unchanged",
    ],
})
display(quality_summary)

,measure,before,after_cleaned_csv,decision
0,rows,1200,1200,preserved
1,columns,20,20,preserved
2,missing cells,138,138,preserved for pipeline
3,complete duplicates,0,0,none found
4,duplicate customer IDs,0,0,none found
5,changed cells,0,0,no correction justified
6,IQR-flagged rows,125,125,retained unchanged


## Phase 2 conclusion

The source remains unchanged, the cleaned CSV preserves every customer and genuine missing value, and the fixed 960/240 stratified split is protected by a dataset fingerprint. The reusable transformer prevents identifier/target leakage and learns imputation, scaling, and encoding only from training data. Cross-field questions and outliers remain unresolved pending business documentation. **Stop here until Phase 3 is approved.**